In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import Embedding, GlobalMaxPooling1D, GRU, Dense
import tensorflow as tf
from tensorflow.keras import layers



class TransformerEncoder(layers.Layer):

    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads,
            dropout=dropout
        )

        self.ffn = tf.keras.Sequential([
            layers.Dense(d_ff, activation="relu"),
            layers.Dense(d_model)
        ])

        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, x, training=False):


        attention_output = self.attention(
            query=x,
            key=x,
            value=x,
            training=training
        )

        x = self.norm1(
            x + self.dropout1(attention_output, training=training)
        )

        ffn_output = self.ffn(x, training=training)

        x = self.norm2(
            x + self.dropout2(ffn_output, training=training)
        )

        return x

class HyModel(keras.Model):

    def __init__( self, vocab_size, seq_length, d_model=64, num_heads=4, d_ff=128, dropout=0.1):
        super().__init__()


        self.embedding_layer = layers.Embedding(vocab_size, d_model)

        self.rnn = layers.GRU(d_model, return_sequences=True, activation="tanh", kernel_initializer="glorot_uniform")

        self.encoder = TransformerEncoder(
            d_model=d_model,
            num_heads=num_heads,
            d_ff=d_ff,
            dropout=dropout
        )

        self.pool=GlobalMaxPooling1D()

        self.classifier = layers.Dense(
            1,
            activation="sigmoid"
        )

    def call(self, x):

        x = self.embedding_layer(x)


        x = self.rnn(x)

        x = self.encoder(x)

        x = self.pool(x)

        return self.classifier(x)


train_model=HyModel(5**km+1,12,d_model,h,d_model,rate)
train_model.compile(optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)
Hotest=train_model.fit(X_train_non_ov, y_train,batch_size=b,  epochs=epoch, validation_split=v, verbose=1, shuffle=True)